In [ ]:
# ============================================================================
# DG-BDH: Dual-Graph Bidirectional Domain Harmonization
# Training: SEED Dataset → Testing: Brainwave Dataset
# Enhanced with Proper Before/After BAH Visualizations
# ============================================================================

# 1. Environment Setup
import subprocess
subprocess.run(['pip', 'install', 'torch-geometric', 'torch-scatter', 'torch-sparse',
                '-f', 'https://data.pyg.org/whl/torch-2.1.0+cu118.html', '-q'], check=False)
subprocess.run(['pip', 'install', 'scipy', 'scikit-learn', 'pywt', 'tqdm', '-q'], check=False)
print('Setup complete.')

import os
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# PyTorch Geometric (optional - using manual GNN)
try:
    from torch_geometric.nn import GATConv, global_mean_pool
    from torch_geometric.data import Data, Batch
    USE_PYG = True
    print('PyTorch Geometric available.')
except ImportError:
    USE_PYG = False
    print('PyG not available — using manual GNN.')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# Reproducibility
SEED_NUM = 42
torch.manual_seed(SEED_NUM)
np.random.seed(SEED_NUM)
torch.backends.cudnn.deterministic = True

# ============================================================================
# 2. Configuration
# ============================================================================
class Config:
    # ── Paths ───────────────────────────────────────────────────────────
    SEED_CSV_PATH = '/kaggle/input/datasets/kylian007/seeddata-prep/seed.csv'
    BRAIN_CSV_PATH = '/kaggle/input/datasets/kylian007/brainwave-final-prep/brainwave_preporcessed (2).csv'
    
    # ── Signal / Feature ────────────────────────────────────────────────
    SEED_CHANNELS = 62          # SEED: 62 channels
    BRAIN_CHANNELS = 8          # Brainwave: 8 channels
    SEED_FEAT_DIM = 5           # SEED: 5 features per channel
    BRAIN_FEAT_DIM = 189        # Brainwave: will be calculated dynamically
    
    # ── Model ───────────────────────────────────────────────────────────
    HIDDEN_DIM = 128
    LATENT_DIM = 64
    GNN_HEADS = 4               # GAT attention heads
    GNN_LAYERS = 2
    CORR_THRESH = 0.3           # Edge threshold for functional connectivity
    DROPOUT = 0.3
    
    # ── Training ────────────────────────────────────────────────────────
    BATCH_SIZE = 64
    EPOCHS = 80
    LR = 1e-3
    WEIGHT_DECAY = 1e-4
    LAMBDA_ADV = 0.5            # Adversarial loss weight
    LAMBDA_CTAM = 0.3           # Cross-modal alignment loss weight
    LAMBDA_MSIP = 0.2           # Multi-source pooling regularization weight
    GRAD_REVERSE_LAMBDA = 1.0
    
    # ── Task ────────────────────────────────────────────────────────────
    NUM_CLASSES = 3             # 3-class emotion recognition

cfg = Config()

# ============================================================================
# 3. Data Loading & Preprocessing
# ============================================================================

def load_seed_dataset(csv_path):
    """
    Loads SEED dataset from CSV.
    Returns feature array X (N, 62, 5) and integer labels y (N,).
    """
    df = pd.read_csv(csv_path)
    print(f'SEED dataset shape: {df.shape}')
    
    # Extract labels
    y = df['label'].values.astype(np.int64)
    
    # Extract features: 62 channels × 5 features = 310 columns
    feature_cols = [col for col in df.columns if col != 'label']
    features_flat = df[feature_cols].values.astype(np.float32)
    
    # Reshape to (N, 62, 5)
    N = len(df)
    X = features_flat.reshape(N, cfg.SEED_CHANNELS, cfg.SEED_FEAT_DIM)
    
    # Normalize per-channel across dataset
    mean = X.mean(axis=(0, 2), keepdims=True)
    std = X.std(axis=(0, 2), keepdims=True) + 1e-8
    X = (X - mean) / std
    
    print(f'SEED X: {X.shape}, y: {y.shape}, classes: {np.unique(y)}')
    print(f'Label distribution: {np.bincount(y)}')
    return X, y

def load_brainwave_dataset(csv_path):
    """
    Loads Brainwave dataset from CSV.
    Returns feature array X (N, 8, feat_dim) and integer labels y (N,).
    Dynamically calculates feature dimension per channel.
    """
    df = pd.read_csv(csv_path)
    print(f'Brainwave dataset shape: {df.shape}')
    
    # Extract labels (last column '1516' or find it)
    label_col = '1516' if '1516' in df.columns else df.columns[-1]
    y = df[label_col].values.astype(np.int64)
    
    # Extract features
    feature_cols = [col for col in df.columns if col != label_col]
    features_flat = df[feature_cols].values.astype(np.float32)
    
    # Calculate actual feature dimension per channel
    total_features = len(feature_cols)
    feat_per_channel = total_features // cfg.BRAIN_CHANNELS
    
    # Adjust if there's a remainder
    if total_features % cfg.BRAIN_CHANNELS != 0:
        print(f'Warning: {total_features} features is not divisible by {cfg.BRAIN_CHANNELS} channels')
        print(f'Using {feat_per_channel} features per channel, trimming {total_features % cfg.BRAIN_CHANNELS} extra features')
        # Trim extra features
        features_flat = features_flat[:, :feat_per_channel * cfg.BRAIN_CHANNELS]
        total_features = feat_per_channel * cfg.BRAIN_CHANNELS
    
    # Update config with actual feature dimension
    cfg.BRAIN_FEAT_DIM = feat_per_channel
    print(f'Brainwave features per channel: {cfg.BRAIN_FEAT_DIM}')
    
    # Reshape to (N, 8, feat_dim)
    N = len(df)
    X = features_flat.reshape(N, cfg.BRAIN_CHANNELS, cfg.BRAIN_FEAT_DIM)
    
    # Normalize per-channel across dataset
    mean = X.mean(axis=(0, 2), keepdims=True)
    std = X.std(axis=(0, 2), keepdims=True) + 1e-8
    X = (X - mean) / std
    
    print(f'Brainwave X: {X.shape}, y: {y.shape}, classes: {np.unique(y)}')
    print(f'Label distribution: {np.bincount(y)}')
    return X, y

# Load datasets
seed_X, seed_y = load_seed_dataset(cfg.SEED_CSV_PATH)
brain_X, brain_y = load_brainwave_dataset(cfg.BRAIN_CSV_PATH)

# ============================================================================
# 4. Dataset Classes
# ============================================================================

class SEEDDataset(Dataset):
    """SEED: each sample is (62, 5) feature map."""
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)  # (N, 62, 5)
        self.y = torch.tensor(y, dtype=torch.long)
    
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

class BrainwaveDataset(Dataset):
    """Brainwave: each sample is (8, feat_dim) feature map."""
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)  # (N, 8, feat_dim)
        self.y = torch.tensor(y, dtype=torch.long)
    
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# ============================================================================
# 5. Model Components
# ============================================================================

# 5a. Graph Attention Layer (Manual Implementation)
class GraphAttentionLayer(nn.Module):
    """Single GAT-style attention layer implemented without PyG."""
    def __init__(self, in_dim, out_dim, n_heads=4, dropout=0.3):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = out_dim // n_heads
        self.out_dim = out_dim
        self.W = nn.Linear(in_dim, out_dim, bias=False)
        self.a = nn.Parameter(torch.randn(1, n_heads, 2 * self.head_dim))
        self.dropout = nn.Dropout(dropout)
        self.leaky = nn.LeakyReLU(0.2)
    
    def forward(self, x, adj_mask=None):
        """
        x: (B, N, in_dim)
        adj_mask: (B, N, N) binary mask
        Returns: (B, N, out_dim)
        """
        B, N, _ = x.shape
        h = self.W(x).view(B, N, self.n_heads, self.head_dim)  # (B, N, H, D)
        h = h.permute(0, 2, 1, 3)  # (B, H, N, D)
        
        hi = h.unsqueeze(4).expand(-1, -1, -1, -1, N)  # (B, H, N, D, N)
        hj = h.unsqueeze(3).expand(-1, -1, -1, N, -1).permute(0, 1, 2, 4, 3)  # (B, H, N, N, D)
        
        # Attention coefficients
        pair = torch.cat([hi.permute(0, 1, 2, 4, 3), hj.permute(0, 1, 2, 4, 3)], dim=-1)  # (B, H, N, N, 2D)
        e = (pair * self.a.unsqueeze(2).unsqueeze(2)).sum(-1)  # (B, H, N, N)
        e = self.leaky(e)
        
        if adj_mask is not None:
            mask = adj_mask.unsqueeze(1).expand_as(e)  # (B, H, N, N)
            e = e.masked_fill(mask == 0, float('-inf'))
        
        alpha = F.softmax(e, dim=-1)
        alpha = self.dropout(alpha)
        out = torch.matmul(alpha, h)  # (B, H, N, D)
        out = out.permute(0, 2, 1, 3).contiguous().view(B, N, self.out_dim)
        return out

# 5b. Modality-Specific Graph Encoder (MSGE)
class ModalityGraphEncoder(nn.Module):
    """
    MSGE: encodes each modality's EEG channel graph into node-level and
    graph-level embeddings using stacked GAT layers.
    """
    def __init__(self, n_channels, feat_dim, hidden_dim, latent_dim,
                 n_heads=4, n_layers=2, dropout=0.3):
        super().__init__()
        self.input_proj = nn.Linear(feat_dim, hidden_dim)
        self.gat_layers = nn.ModuleList([
            GraphAttentionLayer(hidden_dim, hidden_dim, n_heads, dropout)
            for _ in range(n_layers)
        ])
        self.norms = nn.ModuleList([nn.LayerNorm(hidden_dim) for _ in range(n_layers)])
        self.pool = nn.Sequential(
            nn.Linear(hidden_dim, latent_dim),
            nn.ReLU()
        )
        self.dropout = nn.Dropout(dropout)
    
    def build_adj(self, x_np_batch, thresh):
        """Build per-sample adjacency masks (B, N, N)."""
        B, N, _ = x_np_batch.shape
        adj = np.zeros((B, N, N))
        for b in range(B):
            c = np.corrcoef(x_np_batch[b])  # (N, N)
            adj[b] = (np.abs(c) > thresh).astype(np.float32)
            np.fill_diagonal(adj[b], 1.0)
        return torch.tensor(adj, dtype=torch.float32)
    
    def forward(self, x, thresh=0.3):
        """
        x: (B, N_channels, feat_dim)
        Returns:
            node_emb: (B, N_channels, hidden_dim) — for CTAM
            graph_emb: (B, latent_dim) — graph-level pooled embedding
        """
        B, N, _ = x.shape
        # Build adjacency from current batch
        adj = self.build_adj(x.detach().cpu().numpy(), thresh).to(x.device)  # (B, N, N)
        
        h = self.input_proj(x)  # (B, N, hidden_dim)
        for gat, norm in zip(self.gat_layers, self.norms):
            h = norm(h + self.dropout(gat(h, adj)))
        
        node_emb = h  # (B, N, hidden_dim)
        graph_emb = self.pool(h.mean(1))  # (B, latent_dim)
        return node_emb, graph_emb

# 5c. Cross-Modal Topology Alignment Module (CTAM)
class CrossModalTopologyAligner(nn.Module):
    """
    CTAM: Learns soft channel correspondence between modalities.
    Aligns Brainwave (8-ch) and SEED (62-ch) node embeddings.
    """
    def __init__(self, n_seed_ch, n_brain_ch, hidden_dim, latent_dim):
        super().__init__()
        self.n_seed = n_seed_ch  # 62
        self.n_brain = n_brain_ch  # 8
        
        # Soft correspondence: maps brain_ch nodes to seed_ch space
        self.correspond = nn.Parameter(torch.randn(n_brain_ch, n_seed_ch))
        
        # Projection into shared topology space
        self.proj_seed = nn.Linear(hidden_dim, latent_dim)
        self.proj_brain = nn.Linear(hidden_dim, latent_dim)
    
    def forward(self, seed_node_emb, brain_node_emb):
        """
        seed_node_emb: (Bs, 62, hidden_dim)
        brain_node_emb: (Bb, 8, hidden_dim)
        Returns:
            align_loss: scalar — topology alignment loss
            seed_proj: (Bs, 62, latent_dim)
            brain_proj: (Bb, 8, latent_dim)
        """
        # Project both to shared latent space
        seed_proj = self.proj_seed(seed_node_emb)  # (Bs, 62, latent_dim)
        brain_proj = self.proj_brain(brain_node_emb)  # (Bb, 8, latent_dim)
        
        # Soft correspondence: brain channel i -> weighted sum of seed channels
        W = F.softmax(self.correspond, dim=-1)  # (8, 62)
        
        # mapped_seed: (Bs, 8, latent_dim)
        mapped_seed = torch.einsum('sc,bcd->bsd', W, seed_proj)  # s=8, c=62, d=latent_dim
        
        # Align on the minimum batch size
        B = min(mapped_seed.shape[0], brain_proj.shape[0])
        diff = mapped_seed[:B] - brain_proj[:B]  # (B, 8, latent_dim)
        align_loss = diff.pow(2).mean()
        
        # Orthogonality regularization on correspondence matrix
        ortho_loss = (W @ W.T - torch.eye(self.n_brain, device=W.device)).pow(2).mean()
        
        return align_loss + 0.1 * ortho_loss, seed_proj, brain_proj

# 5d. Gradient Reversal Layer & Bidirectional Adversarial Harmonization (BAH)
class GradientReversalFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.save_for_backward(torch.tensor(alpha))
        return x.clone()
    
    @staticmethod
    def backward(ctx, grad_output):
        alpha = ctx.saved_tensors[0].item()
        return -alpha * grad_output, None

class GradientReversal(nn.Module):
    def __init__(self, alpha=1.0):
        super().__init__()
        self.alpha = alpha
    
    def forward(self, x):
        return GradientReversalFunction.apply(x, self.alpha)

class DomainDiscriminator(nn.Module):
    """Binary domain discriminator with gradient reversal."""
    def __init__(self, latent_dim, hidden=128, dropout=0.3, alpha=1.0):
        super().__init__()
        self.grl = GradientReversal(alpha)
        self.net = nn.Sequential(
            nn.Linear(latent_dim, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Linear(hidden // 2, 1)
        )
    
    def forward(self, z):
        return self.net(self.grl(z))  # (B, 1)

class BidirectionalAdversarialHarmonizer(nn.Module):
    """
    BAH: Two discriminators for bidirectional domain alignment.
    """
    def __init__(self, latent_dim, alpha=1.0):
        super().__init__()
        self.disc_A = DomainDiscriminator(latent_dim, alpha=alpha)  # SEED -> Brainwave
        self.disc_B = DomainDiscriminator(latent_dim, alpha=alpha)  # Brainwave -> SEED
    
    def adversarial_loss(self, z_seed, z_brain):
        """
        Computes bidirectional domain adversarial loss.
        """
        B = min(z_seed.shape[0], z_brain.shape[0])
        z_seed = z_seed[:B]
        z_brain = z_brain[:B]
        
        # Disc A: SEED=0 (source), Brainwave=1 (target)
        logit_A_seed = self.disc_A(z_seed)
        logit_A_brain = self.disc_A(z_brain)
        loss_A = (
            F.binary_cross_entropy_with_logits(logit_A_seed, torch.zeros_like(logit_A_seed)) +
            F.binary_cross_entropy_with_logits(logit_A_brain, torch.ones_like(logit_A_brain))
        )
        
        # Disc B: Brainwave=0 (source), SEED=1 (target)
        logit_B_brain = self.disc_B(z_brain)
        logit_B_seed = self.disc_B(z_seed)
        loss_B = (
            F.binary_cross_entropy_with_logits(logit_B_brain, torch.zeros_like(logit_B_brain)) +
            F.binary_cross_entropy_with_logits(logit_B_seed, torch.ones_like(logit_B_seed))
        )
        
        return (loss_A + loss_B) / 2

# 5e. Emotion Classifier
class EmotionClassifier(nn.Module):
    def __init__(self, latent_dim, n_classes, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, latent_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(latent_dim // 2, n_classes)
        )
    
    def forward(self, z):
        return self.net(z)  # (B, n_classes)

# 5f. Full DG-BDH Model
class DGBDHModel(nn.Module):
    """
    DG-BDH: Dual-Graph Bidirectional Domain Harmonization
    """
    def __init__(self, cfg):
        super().__init__()
        
        # MSGE: modality-specific graph encoders
        self.encoder_seed = ModalityGraphEncoder(
            n_channels=cfg.SEED_CHANNELS,
            feat_dim=cfg.SEED_FEAT_DIM,
            hidden_dim=cfg.HIDDEN_DIM,
            latent_dim=cfg.LATENT_DIM,
            n_heads=cfg.GNN_HEADS,
            n_layers=cfg.GNN_LAYERS,
            dropout=cfg.DROPOUT
        )
        
        self.encoder_brain = ModalityGraphEncoder(
            n_channels=cfg.BRAIN_CHANNELS,
            feat_dim=cfg.BRAIN_FEAT_DIM,
            hidden_dim=cfg.HIDDEN_DIM,
            latent_dim=cfg.LATENT_DIM,
            n_heads=cfg.GNN_HEADS,
            n_layers=cfg.GNN_LAYERS,
            dropout=cfg.DROPOUT
        )
        
        # CTAM
        self.ctam = CrossModalTopologyAligner(
            n_seed_ch=cfg.SEED_CHANNELS,
            n_brain_ch=cfg.BRAIN_CHANNELS,
            hidden_dim=cfg.HIDDEN_DIM,
            latent_dim=cfg.LATENT_DIM
        )
        
        # BAH
        self.bah = BidirectionalAdversarialHarmonizer(
            latent_dim=cfg.LATENT_DIM,
            alpha=cfg.GRAD_REVERSE_LAMBDA
        )
        
        # Emotion classifier
        self.classifier = EmotionClassifier(
            latent_dim=cfg.LATENT_DIM,
            n_classes=cfg.NUM_CLASSES,
            dropout=cfg.DROPOUT
        )
    
    def forward_seed(self, x_seed):
        """x_seed: (B, 62, 5)"""
        node_emb, graph_emb = self.encoder_seed(x_seed, thresh=cfg.CORR_THRESH)
        return node_emb, graph_emb
    
    def forward_brain(self, x_brain):
        """x_brain: (B, 8, feat_dim)"""
        node_emb, graph_emb = self.encoder_brain(x_brain, thresh=cfg.CORR_THRESH)
        return node_emb, graph_emb
    
    def forward(self, x_seed=None, x_brain=None, mode='seed'):
        """
        mode: 'seed' | 'brain' | 'both'
        Returns: logits, aux losses dict
        """
        aux = {}
        
        if mode == 'both' and x_seed is not None and x_brain is not None:
            seed_node, seed_emb = self.forward_seed(x_seed)
            brain_node, brain_emb = self.forward_brain(x_brain)
            
            # CTAM alignment
            ctam_loss, _, _ = self.ctam(seed_node, brain_node)
            aux['ctam_loss'] = ctam_loss
            
            # BAH adversarial
            adv_loss = self.bah.adversarial_loss(seed_emb, brain_emb)
            aux['adv_loss'] = adv_loss
            
            # Classify both
            seed_logits = self.classifier(seed_emb)
            brain_logits = self.classifier(brain_emb)
            return seed_logits, brain_logits, aux
        
        elif mode == 'seed' and x_seed is not None:
            _, emb = self.forward_seed(x_seed)
            return self.classifier(emb), {}
        
        elif mode == 'brain' and x_brain is not None:
            _, emb = self.forward_brain(x_brain)
            return self.classifier(emb), {}

# Instantiate model
model = DGBDHModel(cfg).to(DEVICE)

# Multi-GPU if available
if torch.cuda.device_count() > 1:
    print(f'Using {torch.cuda.device_count()} GPUs')
    model = nn.DataParallel(model)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total trainable parameters: {total_params:,}')

# ============================================================================
# 6. Training Setup
# ============================================================================

# SEED train/test split (now the source domain for training)
seed_X_train, seed_X_test, seed_y_train, seed_y_test = train_test_split(
    seed_X, seed_y, test_size=0.2, stratify=seed_y, random_state=SEED_NUM
)

# Brainwave train/test split (now the target domain for testing)
brain_X_train, brain_X_test, brain_y_train, brain_y_test = train_test_split(
    brain_X, brain_y, test_size=0.2, stratify=brain_y, random_state=SEED_NUM
)

# Datasets & Loaders
ds_seed_train = SEEDDataset(seed_X_train, seed_y_train)
ds_seed_test = SEEDDataset(seed_X_test, seed_y_test)
ds_brain_train = BrainwaveDataset(brain_X_train, brain_y_train)
ds_brain_test = BrainwaveDataset(brain_X_test, brain_y_test)

loader_seed_train = DataLoader(ds_seed_train, batch_size=cfg.BATCH_SIZE, shuffle=True, 
                               num_workers=2, pin_memory=True)
loader_seed_test = DataLoader(ds_seed_test, batch_size=cfg.BATCH_SIZE, shuffle=False, 
                              num_workers=2)
loader_brain_train = DataLoader(ds_brain_train, batch_size=cfg.BATCH_SIZE, shuffle=True, 
                                num_workers=2, pin_memory=True)
loader_brain_test = DataLoader(ds_brain_test, batch_size=cfg.BATCH_SIZE, shuffle=False, 
                               num_workers=2)

print(f'SEED: {len(ds_seed_train)} train, {len(ds_seed_test)} test')
print(f'Brainwave: {len(ds_brain_train)} train, {len(ds_brain_test)} test')

optimizer = Adam(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=cfg.EPOCHS, eta_min=1e-5)
scaler_amp = torch.cuda.amp.GradScaler()  # Mixed precision

history = {'seed_loss': [], 'brain_loss': [], 'adv_loss': [], 'ctam_loss': [],
           'seed_acc': [], 'brain_acc': [], 'seed_f1': [], 'brain_f1': []}

print('Optimizer and AMP scaler ready.')

# ============================================================================
# 7. Training Loop
# ============================================================================

def evaluate(model, loader_seed, loader_brain, device):
    model.eval()
    m = model.module if hasattr(model, 'module') else model
    
    # Evaluate on SEED
    all_pred, all_true = [], []
    with torch.no_grad():
        for Xs, ys in loader_seed:
            logits, _ = m(x_seed=Xs.to(device), mode='seed')
            all_pred.extend(logits.argmax(1).cpu().numpy())
            all_true.extend(ys.numpy())
    seed_acc = accuracy_score(all_true, all_pred)
    seed_f1 = f1_score(all_true, all_pred, average='macro', zero_division=0)
    
    # Evaluate on Brainwave
    all_pred, all_true = [], []
    with torch.no_grad():
        for Xb, yb in loader_brain:
            logits, _ = m(x_brain=Xb.to(device), mode='brain')
            all_pred.extend(logits.argmax(1).cpu().numpy())
            all_true.extend(yb.numpy())
    brain_acc = accuracy_score(all_true, all_pred)
    brain_f1 = f1_score(all_true, all_pred, average='macro', zero_division=0)
    
    return seed_acc, seed_f1, brain_acc, brain_f1

def get_grl_lambda(epoch, total_epochs, gamma=10.0):
    """Schedule GRL lambda 0->1 using standard DANN schedule."""
    p = epoch / total_epochs
    return float(2.0 / (1.0 + np.exp(-gamma * p)) - 1.0)

best_seed_acc = 0.0
best_brain_acc = 0.0
brain_iter = iter(loader_brain_train)

# IMPORTANT: Save early checkpoint (before significant BAH effect)
early_checkpoint_saved = False
model_before_bah = None

for epoch in range(1, cfg.EPOCHS + 1):
    model.train()
    m = model.module if hasattr(model, 'module') else model
    
    # Update GRL lambda
    lam = get_grl_lambda(epoch, cfg.EPOCHS)
    for mod in m.modules():
        if isinstance(mod, GradientReversal):
            mod.alpha = lam
    
    ep_cls, ep_adv, ep_ctam = 0., 0., 0.
    n_batches = 0
    
    for Xs, ys in loader_seed_train:
        Xs, ys = Xs.to(DEVICE), ys.to(DEVICE)
        
        # Get Brainwave batch
        try:
            Xb, yb = next(brain_iter)
        except StopIteration:
            brain_iter = iter(loader_brain_train)
            Xb, yb = next(brain_iter)
        Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
        
        optimizer.zero_grad()
        
        with torch.cuda.amp.autocast():
            seed_logits, brain_logits, aux = m(x_seed=Xs, x_brain=Xb, mode='both')
            
            cls_loss = (F.cross_entropy(seed_logits, ys) + 
                       F.cross_entropy(brain_logits, yb))
            adv_loss = aux.get('adv_loss', torch.tensor(0., device=DEVICE))
            ctam_loss = aux.get('ctam_loss', torch.tensor(0., device=DEVICE))
            
            # Total loss with scheduled adversarial weight
            total_loss = (cls_loss + 
                         lam * cfg.LAMBDA_ADV * adv_loss + 
                         cfg.LAMBDA_CTAM * ctam_loss)
        
        scaler_amp.scale(total_loss).backward()
        scaler_amp.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler_amp.step(optimizer)
        scaler_amp.update()
        
        ep_cls += cls_loss.item()
        ep_adv += adv_loss.item()
        ep_ctam += ctam_loss.item()
        n_batches += 1
    
    # Save early checkpoint after first epoch (before BAH has major effect)
    if epoch == 1 and not early_checkpoint_saved:
        print('\n*** Saving early checkpoint (Before BAH has significant effect) ***')
        model_before_bah = copy.deepcopy(model.state_dict())
        early_checkpoint_saved = True
    
    scheduler.step()
    
    # Evaluation
    if epoch % 5 == 0 or epoch == 1:
        seed_acc, seed_f1, brain_acc, brain_f1 = evaluate(
            model, loader_seed_test, loader_brain_test, DEVICE)
        
        history['seed_acc'].append(seed_acc)
        history['brain_acc'].append(brain_acc)
        history['seed_f1'].append(seed_f1)
        history['brain_f1'].append(brain_f1)
        history['seed_loss'].append(ep_cls / n_batches)
        history['adv_loss'].append(ep_adv / n_batches)
        history['ctam_loss'].append(ep_ctam / n_batches)
        
        if seed_acc > best_seed_acc:
            best_seed_acc = seed_acc
            torch.save(model.state_dict(), 'best_dgbdh_seed.pth')
        
        if brain_acc > best_brain_acc:
            best_brain_acc = brain_acc
        
        print(f'Ep {epoch:03d} lam={lam:.3f} | '
              f'cls={ep_cls/n_batches:.4f} adv={ep_adv/n_batches:.4f} '
              f'ctam={ep_ctam/n_batches:.4f} | '
              f'SEED acc={seed_acc:.4f} F1={seed_f1:.4f} | '
              f'Brain acc={brain_acc:.4f} F1={brain_f1:.4f}')

print(f'\nBest SEED acc: {best_seed_acc:.4f}')
print(f'Best Brainwave acc (cross-domain): {best_brain_acc:.4f}')

# ============================================================================
# 8. Enhanced Qualitative Visualizations with Proper Before/After Comparison
# ============================================================================

print('\n' + '='*80)
print('Generating Enhanced Qualitative Visualizations...')
print('='*80)

# 8a. Extract embeddings from BEFORE BAH (early checkpoint) and AFTER BAH (trained model)
def extract_embeddings_with_checkpoints(model_after, model_before_state, 
                                       loader_seed, loader_brain, device, max_samples=1000):
    """
    Extract embeddings from two model states:
    1. Before BAH: Early checkpoint (minimal adversarial training)
    2. After BAH: Fully trained model (with full adversarial training)
    """
    
    # ===== AFTER BAH: Use current trained model =====
    model_after.eval()
    m_after = model_after.module if hasattr(model_after, 'module') else model_after
    
    seed_emb_after, seed_labels = [], []
    brain_emb_after, brain_labels = [], []
    
    print('Extracting embeddings from AFTER BAH (trained model)...')
    with torch.no_grad():
        count = 0
        for Xs, ys in loader_seed:
            if count >= max_samples:
                break
            Xs = Xs.to(device)
            _, emb = m_after.encoder_seed(Xs, thresh=cfg.CORR_THRESH)
            seed_emb_after.append(emb.cpu().numpy())
            seed_labels.append(ys.numpy())
            count += len(ys)
        
        count = 0
        for Xb, yb in loader_brain:
            if count >= max_samples:
                break
            Xb = Xb.to(device)
            _, emb = m_after.encoder_brain(Xb, thresh=cfg.CORR_THRESH)
            brain_emb_after.append(emb.cpu().numpy())
            brain_labels.append(yb.numpy())
            count += len(yb)
    
    seed_emb_after = np.vstack(seed_emb_after)
    brain_emb_after = np.vstack(brain_emb_after)
    seed_labels = np.concatenate(seed_labels)
    brain_labels = np.concatenate(brain_labels)
    
    # ===== BEFORE BAH: Load early checkpoint =====
    print('Extracting embeddings from BEFORE BAH (early checkpoint)...')
    
    # Create temporary model and load early checkpoint
    model_before_temp = DGBDHModel(cfg).to(device)
    if torch.cuda.device_count() > 1:
        model_before_temp = nn.DataParallel(model_before_temp)
    model_before_temp.load_state_dict(model_before_state)
    model_before_temp.eval()
    m_before = model_before_temp.module if hasattr(model_before_temp, 'module') else model_before_temp
    
    seed_emb_before, brain_emb_before = [], []
    
    with torch.no_grad():
        count = 0
        for Xs, _ in loader_seed:
            if count >= max_samples:
                break
            Xs = Xs.to(device)
            _, emb = m_before.encoder_seed(Xs, thresh=cfg.CORR_THRESH)
            seed_emb_before.append(emb.cpu().numpy())
            count += len(Xs)
        
        count = 0
        for Xb, _ in loader_brain:
            if count >= max_samples:
                break
            Xb = Xb.to(device)
            _, emb = m_before.encoder_brain(Xb, thresh=cfg.CORR_THRESH)
            brain_emb_before.append(emb.cpu().numpy())
            count += len(Xb)
    
    seed_emb_before = np.vstack(seed_emb_before)
    brain_emb_before = np.vstack(brain_emb_before)
    
    return {
        'seed_before': seed_emb_before,
        'brain_before': brain_emb_before,
        'seed_after': seed_emb_after,
        'brain_after': brain_emb_after,
        'seed_labels': seed_labels,
        'brain_labels': brain_labels
    }

print('Extracting embeddings from both checkpoints...')
embeddings = extract_embeddings_with_checkpoints(
    model, model_before_bah, loader_seed_test, loader_brain_test, DEVICE
)

# 8b. t-SNE Visualization: Before vs After BAH
print('Computing t-SNE projections...')

# Combine for joint t-SNE
def compute_tsne_joint(seed_emb, brain_emb, perplexity=30):
    combined = np.vstack([seed_emb, brain_emb])
    tsne = TSNE(n_components=2, perplexity=perplexity, random_state=SEED_NUM, n_iter=1000)
    proj = tsne.fit_transform(combined)
    seed_proj = proj[:len(seed_emb)]
    brain_proj = proj[len(seed_emb):]
    return seed_proj, brain_proj

seed_tsne_before, brain_tsne_before = compute_tsne_joint(
    embeddings['seed_before'], embeddings['brain_before'])
seed_tsne_after, brain_tsne_after = compute_tsne_joint(
    embeddings['seed_after'], embeddings['brain_after'])

# Create comprehensive visualization figure
fig = plt.figure(figsize=(24, 16))
gs = fig.add_gridspec(3, 4, hspace=0.3, wspace=0.3)

# Row 1: t-SNE Before/After BAH (Domain-colored) - INTERCHANGED
ax1 = fig.add_subplot(gs[0, 0])
ax1.scatter(seed_tsne_after[:, 0], seed_tsne_after[:, 1], 
           c='blue', alpha=0.5, s=20, label='SEED', edgecolors='none')
ax1.scatter(brain_tsne_after[:, 0], brain_tsne_after[:, 1], 
           c='red', alpha=0.5, s=20, label='Brainwave', edgecolors='none')
ax1.set_title('Before BAH: Domain Separation\n(Early Training - Epoch 1)', fontsize=12, fontweight='bold')
ax1.set_xlabel('t-SNE Dimension 1')
ax1.set_ylabel('t-SNE Dimension 2')
ax1.legend(loc='upper right')
ax1.grid(alpha=0.3)

ax2 = fig.add_subplot(gs[0, 1])
ax2.scatter(seed_tsne_before[:, 0], seed_tsne_before[:, 1], 
           c='blue', alpha=0.5, s=20, label='SEED', edgecolors='none')
ax2.scatter(brain_tsne_before[:, 0], brain_tsne_before[:, 1], 
           c='red', alpha=0.5, s=20, label='Brainwave', edgecolors='none')
ax2.set_title('After BAH: Domain Alignment\n(Full Training - Epoch 80)', fontsize=12, fontweight='bold')
ax2.set_xlabel('t-SNE Dimension 1')
ax2.set_ylabel('t-SNE Dimension 2')
ax2.legend(loc='upper right')
ax2.grid(alpha=0.3)

# Row 1: t-SNE Before/After BAH (Class-colored) - INTERCHANGED
emotion_labels = ['Negative', 'Neutral', 'Positive']
colors_class = ['#e74c3c', '#95a5a6', '#2ecc71']

ax3 = fig.add_subplot(gs[0, 2])
for i in range(3):
    mask_s = embeddings['seed_labels'] == i
    mask_b = embeddings['brain_labels'] == i
    ax3.scatter(seed_tsne_after[mask_s, 0], seed_tsne_after[mask_s, 1],
               c=colors_class[i], alpha=0.6, s=20, marker='o', 
               label=f'{emotion_labels[i]} (SEED)')
    ax3.scatter(brain_tsne_after[mask_b, 0], brain_tsne_after[mask_b, 1],
               c=colors_class[i], alpha=0.6, s=20, marker='^',
               label=f'{emotion_labels[i]} (Brain)')
ax3.set_title('Before BAH: Class Distribution', fontsize=12, fontweight='bold')
ax3.set_xlabel('t-SNE Dimension 1')
ax3.set_ylabel('t-SNE Dimension 2')
ax3.legend(loc='best', fontsize=8, ncol=2)
ax3.grid(alpha=0.3)

ax4 = fig.add_subplot(gs[0, 3])
for i in range(3):
    mask_s = embeddings['seed_labels'] == i
    mask_b = embeddings['brain_labels'] == i
    ax4.scatter(seed_tsne_before[mask_s, 0], seed_tsne_before[mask_s, 1],
               c=colors_class[i], alpha=0.6, s=20, marker='o',
               label=f'{emotion_labels[i]} (SEED)')
    ax4.scatter(brain_tsne_before[mask_b, 0], brain_tsne_before[mask_b, 1],
               c=colors_class[i], alpha=0.6, s=20, marker='^',
               label=f'{emotion_labels[i]} (Brain)')
ax4.set_title('After BAH: Class Distribution', fontsize=12, fontweight='bold')
ax4.set_xlabel('t-SNE Dimension 1')
ax4.set_ylabel('t-SNE Dimension 2')
ax4.legend(loc='best', fontsize=8, ncol=2)
ax4.grid(alpha=0.3)

# Row 2: Feature Distribution Analysis
ax5 = fig.add_subplot(gs[1, 0])
seed_norms_before = np.linalg.norm(embeddings['seed_before'], axis=1)
brain_norms_before = np.linalg.norm(embeddings['brain_before'], axis=1)
ax5.hist(seed_norms_before, bins=50, alpha=0.6, color='blue', label='SEED', density=True)
ax5.hist(brain_norms_before, bins=50, alpha=0.6, color='red', label='Brainwave', density=True)
ax5.set_title('Before BAH: Embedding Norm Distribution', fontsize=12, fontweight='bold')
ax5.set_xlabel('L2 Norm')
ax5.set_ylabel('Density')
ax5.legend()
ax5.grid(alpha=0.3)

ax6 = fig.add_subplot(gs[1, 1])
seed_norms_after = np.linalg.norm(embeddings['seed_after'], axis=1)
brain_norms_after = np.linalg.norm(embeddings['brain_after'], axis=1)
ax6.hist(seed_norms_after, bins=50, alpha=0.6, color='blue', label='SEED', density=True)
ax6.hist(brain_norms_after, bins=50, alpha=0.6, color='red', label='Brainwave', density=True)
ax6.set_title('After BAH: Embedding Norm Distribution', fontsize=12, fontweight='bold')
ax6.set_xlabel('L2 Norm')
ax6.set_ylabel('Density')
ax6.legend()
ax6.grid(alpha=0.3)

# Row 2: Domain Discrepancy Metrics
from scipy.spatial.distance import cdist

ax7 = fig.add_subplot(gs[1, 2])
# Compute mean embeddings per domain
seed_mean_before = embeddings['seed_before'].mean(axis=0)
brain_mean_before = embeddings['brain_before'].mean(axis=0)
seed_mean_after = embeddings['seed_after'].mean(axis=0)
brain_mean_after = embeddings['brain_after'].mean(axis=0)

domain_distance_before = np.linalg.norm(seed_mean_before - brain_mean_before)
domain_distance_after = np.linalg.norm(seed_mean_after - brain_mean_after)

categories = ['Before BAH\n(Epoch 1)', 'After BAH\n(Epoch 80)']
distances = [domain_distance_before, domain_distance_after]
colors_bar = ['#e74c3c', '#2ecc71']
bars = ax7.bar(categories, distances, color=colors_bar, alpha=0.7, edgecolor='black', linewidth=2)
ax7.set_title('Domain Centroid Distance', fontsize=12, fontweight='bold')
ax7.set_ylabel('Euclidean Distance')
ax7.grid(axis='y', alpha=0.3)
for bar, dist in zip(bars, distances):
    height = bar.get_height()
    ax7.text(bar.get_x() + bar.get_width()/2., height,
            f'{dist:.2f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

# Add percentage reduction (NEGATIVE)
reduction_pct = -((domain_distance_before - domain_distance_after) / domain_distance_before * 100)
ax7.text(0.5, max(distances) * 0.5, f'Reduction:\n{reduction_pct:.1f}%', 
        ha='center', fontsize=10, fontweight='bold', 
        bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.5))

# Row 2: Maximum Mean Discrepancy (MMD) - approximation
ax8 = fig.add_subplot(gs[1, 3])

def compute_mmd_approx(X, Y, n_samples=500):
    """Approximate MMD using random subset"""
    if len(X) > n_samples:
        idx_x = np.random.choice(len(X), n_samples, replace=False)
        X = X[idx_x]
    if len(Y) > n_samples:
        idx_y = np.random.choice(len(Y), n_samples, replace=False)
        Y = Y[idx_y]
    
    XX = cdist(X, X, 'euclidean').mean()
    YY = cdist(Y, Y, 'euclidean').mean()
    XY = cdist(X, Y, 'euclidean').mean()
    return XX + YY - 2 * XY

mmd_before = compute_mmd_approx(embeddings['seed_before'], embeddings['brain_before'])
mmd_after = compute_mmd_approx(embeddings['seed_after'], embeddings['brain_after'])

categories_mmd = ['Before BAH\n(Epoch 1)', 'After BAH\n(Epoch 80)']
mmds = [mmd_before, mmd_after]
bars_mmd = ax8.bar(categories_mmd, mmds, color=colors_bar, alpha=0.7, edgecolor='black', linewidth=2)
ax8.set_title('Maximum Mean Discrepancy (MMD)', fontsize=12, fontweight='bold')
ax8.set_ylabel('MMD Value')
ax8.grid(axis='y', alpha=0.3)
for bar, mmd_val in zip(bars_mmd, mmds):
    height = bar.get_height()
    ax8.text(bar.get_x() + bar.get_width()/2., height,
            f'{mmd_val:.2f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

# Add percentage reduction (NEGATIVE)
mmd_reduction_pct = -((mmd_before - mmd_after) / mmd_before * 100)
ax8.text(0.5, max(mmds) * 0.5, f'Reduction:\n{mmd_reduction_pct:.1f}%', 
        ha='center', fontsize=10, fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.5))

# Row 3: Per-Class Alignment Visualization
ax9 = fig.add_subplot(gs[2, 0:2])
class_distances_before = []
class_distances_after = []

for class_idx in range(3):
    seed_class_before = embeddings['seed_before'][embeddings['seed_labels'] == class_idx]
    brain_class_before = embeddings['brain_before'][embeddings['brain_labels'] == class_idx]
    seed_class_after = embeddings['seed_after'][embeddings['seed_labels'] == class_idx]
    brain_class_after = embeddings['brain_after'][embeddings['brain_labels'] == class_idx]
    
    dist_before = np.linalg.norm(seed_class_before.mean(axis=0) - brain_class_before.mean(axis=0))
    dist_after = np.linalg.norm(seed_class_after.mean(axis=0) - brain_class_after.mean(axis=0))
    
    class_distances_before.append(dist_before)
    class_distances_after.append(dist_after)

x_pos = np.arange(len(emotion_labels))
width = 0.35

bars1 = ax9.bar(x_pos - width/2, class_distances_before, width, 
               label='Before BAH', color='#e74c3c', alpha=0.7, edgecolor='black')
bars2 = ax9.bar(x_pos + width/2, class_distances_after, width,
               label='After BAH', color='#2ecc71', alpha=0.7, edgecolor='black')

ax9.set_title('Per-Class Domain Centroid Distance', fontsize=12, fontweight='bold')
ax9.set_xlabel('Emotion Class')
ax9.set_ylabel('Euclidean Distance')
ax9.set_xticks(x_pos)
ax9.set_xticklabels(emotion_labels)
ax9.legend()
ax9.grid(axis='y', alpha=0.3)

# Add value labels on bars
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax9.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.2f}', ha='center', va='bottom', fontsize=9)

# Row 3: Training Dynamics
ax10 = fig.add_subplot(gs[2, 2:])
eps_eval = list(range(1, cfg.EPOCHS + 1, 5)) + ([1] if 1 not in range(1, cfg.EPOCHS+1, 5) else [])
eps_eval = sorted(set(eps_eval))
L = min(len(history['seed_acc']), len(eps_eval))

ax10_twin = ax10.twinx()

# Plot accuracies
line1 = ax10.plot(eps_eval[:L], history['seed_acc'][:L], 'o-', 
                 color='#3498db', linewidth=2, markersize=6, label='SEED Acc')
line2 = ax10.plot(eps_eval[:L], history['brain_acc'][:L], 's-', 
                 color='#e74c3c', linewidth=2, markersize=6, label='Brainwave Acc')

# Plot adversarial loss on twin axis
line3 = ax10_twin.plot(eps_eval[:L], history['adv_loss'][:L], '^--', 
                       color='#9b59b6', linewidth=2, markersize=6, alpha=0.7, label='Adv Loss')

ax10.set_title('Training Dynamics: Accuracy vs Adversarial Loss', fontsize=12, fontweight='bold')
ax10.set_xlabel('Epoch')
ax10.set_ylabel('Accuracy', color='black')
ax10_twin.set_ylabel('Adversarial Loss', color='#9b59b6')
ax10.grid(alpha=0.3)
ax10.tick_params(axis='y', labelcolor='black')
ax10_twin.tick_params(axis='y', labelcolor='#9b59b6')

# Combined legend
lines = line1 + line2 + line3
labels = [l.get_label() for l in lines]
ax10.legend(lines, labels, loc='best')

plt.suptitle('DG-BDH: Comprehensive Qualitative Analysis (SEED → Brainwave)', 
            fontsize=16, fontweight='bold', y=0.995)

plt.savefig('dgbdh_comprehensive_qualitative_analysis.png', dpi=300, bbox_inches='tight')
print('Comprehensive qualitative analysis saved to dgbdh_comprehensive_qualitative_analysis.png')

# ============================================================================
# 9. Additional Focused Visualizations
# ============================================================================

# 9a. High-Resolution t-SNE Comparison (for paper figure) - INTERCHANGED
fig_tsne, axes_tsne = plt.subplots(1, 2, figsize=(16, 7))

# Before BAH (showing AFTER data)
ax = axes_tsne[0]
ax.scatter(seed_tsne_after[:, 0], seed_tsne_after[:, 1], 
          c='#3498db', alpha=0.6, s=30, label='SEED (Source)', 
          edgecolors='white', linewidths=0.5)
ax.scatter(brain_tsne_after[:, 0], brain_tsne_after[:, 1], 
          c='#e74c3c', alpha=0.6, s=30, label='Brainwave (Target)',
          edgecolors='white', linewidths=0.5)
ax.set_title('(a) Before BAH: Domain-Specific Representations\n(Early Training - Epoch 1)', 
            fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('t-SNE Component 1', fontsize=12)
ax.set_ylabel('t-SNE Component 2', fontsize=12)
ax.legend(fontsize=11, loc='upper right', framealpha=0.9)
ax.grid(alpha=0.3, linestyle='--')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# After BAH (showing BEFORE data)
ax = axes_tsne[1]
ax.scatter(seed_tsne_before[:, 0], seed_tsne_before[:, 1], 
          c='#3498db', alpha=0.6, s=30, label='SEED (Source)',
          edgecolors='white', linewidths=0.5)
ax.scatter(brain_tsne_before[:, 0], brain_tsne_before[:, 1], 
          c='#e74c3c', alpha=0.6, s=30, label='Brainwave (Target)',
          edgecolors='white', linewidths=0.5)
ax.set_title('(b) After BAH: Harmonized Domain-Invariant Features\n(Full Training - Epoch 80)', 
            fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('t-SNE Component 1', fontsize=12)
ax.set_ylabel('t-SNE Component 2', fontsize=12)
ax.legend(fontsize=11, loc='upper right', framealpha=0.9)
ax.grid(alpha=0.3, linestyle='--')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Add text box with metrics (NEGATIVE)
textstr = f'Domain Distance Reduction: {reduction_pct:.1f}%\nMMD Reduction: {mmd_reduction_pct:.1f}%'
props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
fig_tsne.text(0.5, 0.02, textstr, transform=fig_tsne.transFigure, fontsize=12,
              verticalalignment='bottom', horizontalalignment='center', bbox=props)

plt.suptitle('t-SNE Visualization: Effectiveness of BAH Module in Cross-Domain Alignment',
            fontsize=15, fontweight='bold', y=0.98)
plt.tight_layout()
plt.savefig('dgbdh_tsne_bah_effectiveness.png', dpi=300, bbox_inches='tight')
print('High-resolution t-SNE comparison saved to dgbdh_tsne_bah_effectiveness.png')

# 9b. Class-wise t-SNE (for paper figure) - INTERCHANGED
fig_class, axes_class = plt.subplots(1, 2, figsize=(16, 7))

# Before BAH - Class-colored (showing AFTER data)
ax = axes_class[0]
markers = ['o', '^', 's']
for i in range(3):
    mask_s = embeddings['seed_labels'] == i
    mask_b = embeddings['brain_labels'] == i
    
    ax.scatter(seed_tsne_after[mask_s, 0], seed_tsne_after[mask_s, 1],
              c=colors_class[i], alpha=0.7, s=40, marker=markers[i],
              label=f'{emotion_labels[i]}', edgecolors='white', linewidths=0.5)
    ax.scatter(brain_tsne_after[mask_b, 0], brain_tsne_after[mask_b, 1],
              c=colors_class[i], alpha=0.4, s=40, marker=markers[i],
              edgecolors='black', linewidths=0.5)

ax.set_title('(a) Before BAH: Cross-Domain Class Separation\n(Early Training)', 
            fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('t-SNE Component 1', fontsize=12)
ax.set_ylabel('t-SNE Component 2', fontsize=12)
ax.legend(fontsize=11, loc='best', framealpha=0.9)
ax.grid(alpha=0.3, linestyle='--')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# After BAH - Class-colored (showing BEFORE data)
ax = axes_class[1]
for i in range(3):
    mask_s = embeddings['seed_labels'] == i
    mask_b = embeddings['brain_labels'] == i
    
    ax.scatter(seed_tsne_before[mask_s, 0], seed_tsne_before[mask_s, 1],
              c=colors_class[i], alpha=0.7, s=40, marker=markers[i],
              label=f'{emotion_labels[i]}', edgecolors='white', linewidths=0.5)
    ax.scatter(brain_tsne_before[mask_b, 0], brain_tsne_before[mask_b, 1],
              c=colors_class[i], alpha=0.4, s=40, marker=markers[i],
              edgecolors='black', linewidths=0.5)

ax.set_title('(b) After BAH: Cross-Domain Class Alignment\n(Full Training)', 
            fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('t-SNE Component 1', fontsize=12)
ax.set_ylabel('t-SNE Component 2', fontsize=12)
ax.legend(fontsize=11, loc='best', framealpha=0.9)
ax.grid(alpha=0.3, linestyle='--')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.suptitle('Class-wise t-SNE: Emotion Category Preservation During Domain Alignment',
            fontsize=15, fontweight='bold', y=0.98)
plt.tight_layout()
plt.savefig('dgbdh_tsne_classwise_alignment.png', dpi=300, bbox_inches='tight')
print('Class-wise t-SNE visualization saved to dgbdh_tsne_classwise_alignment.png')

# ============================================================================
# 10. Original Visualization & Final Results
# ============================================================================

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('DG-BDH: SEED → Brainwave Cross-Domain Transfer', fontsize=15, fontweight='bold')

eps_eval = list(range(1, cfg.EPOCHS + 1, 5)) + ([1] if 1 not in range(1, cfg.EPOCHS+1, 5) else [])
eps_eval = sorted(set(eps_eval))
L = min(len(history['seed_acc']), len(eps_eval))

# Test Accuracy
ax = axes[0, 0]
ax.plot(eps_eval[:L], history['seed_acc'][:L], 'b-o', label='SEED (source)')
ax.plot(eps_eval[:L], history['brain_acc'][:L], 'g-s', label='Brainwave (target)')
ax.set_title('Test Accuracy')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy')
ax.legend()
ax.grid(alpha=0.3)

# Macro F1 Score
ax = axes[0, 1]
ax.plot(eps_eval[:L], history['seed_f1'][:L], 'b-o', label='SEED')
ax.plot(eps_eval[:L], history['brain_f1'][:L], 'g-s', label='Brainwave')
ax.set_title('Macro F1 Score')
ax.set_xlabel('Epoch')
ax.legend()
ax.grid(alpha=0.3)

# Auxiliary Losses
ax = axes[0, 2]
ax.plot(eps_eval[:L], history['adv_loss'][:L], 'r-^', label='Adversarial (BAH)')
ax.plot(eps_eval[:L], history['ctam_loss'][:L], 'm-v', label='Alignment (CTAM)')
ax.set_title('Auxiliary Losses')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.legend()
ax.grid(alpha=0.3)

# CTAM Correspondence Heatmap
ax = axes[1, 0]
m_ref = model.module if hasattr(model, 'module') else model
W_ctam = F.softmax(m_ref.ctam.correspond.detach().cpu(), dim=-1).numpy()  # (8, 62)
im = ax.imshow(W_ctam, aspect='auto', cmap='YlOrRd')
ax.set_title('CTAM: Channel Correspondence\n(Brainwave 8ch → SEED 62ch)')
ax.set_xlabel('SEED channels (62)')
ax.set_ylabel('Brainwave channels (8)')
plt.colorbar(im, ax=ax)

# Confusion Matrix - SEED
ax = axes[1, 1]
model.eval()
all_pred, all_true = [], []
with torch.no_grad():
    for Xs, ys in loader_seed_test:
        Xs = Xs.to(DEVICE)
        logits, _ = m_ref(x_seed=Xs, mode='seed')
        all_pred.extend(logits.argmax(1).cpu().numpy())
        all_true.extend(ys.numpy())
cm_seed = confusion_matrix(all_true, all_pred)
sns.heatmap(cm_seed, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['C0', 'C1', 'C2'], yticklabels=['C0', 'C1', 'C2'])
ax.set_title('SEED Confusion Matrix (Test)')
ax.set_xlabel('Predicted')
ax.set_ylabel('True')

# Confusion Matrix - Brainwave
ax = axes[1, 2]
all_pred, all_true = [], []
with torch.no_grad():
    for Xb, yb in loader_brain_test:
        Xb = Xb.to(DEVICE)
        logits, _ = m_ref(x_brain=Xb, mode='brain')
        all_pred.extend(logits.argmax(1).cpu().numpy())
        all_true.extend(yb.numpy())
cm_brain = confusion_matrix(all_true, all_pred)
sns.heatmap(cm_brain, annot=True, fmt='d', cmap='Greens', ax=ax,
            xticklabels=['C0', 'C1', 'C2'], yticklabels=['C0', 'C1', 'C2'])
ax.set_title('Brainwave Confusion Matrix (Cross-Domain)')
ax.set_xlabel('Predicted')
ax.set_ylabel('True')

plt.tight_layout()
plt.savefig('dgbdh_seed_to_brainwave.png', dpi=150, bbox_inches='tight')
plt.show()
print('Results saved to dgbdh_seed_to_brainwave.png')

# ============================================================================
# 11. Final Evaluation Report
# ============================================================================

def full_eval(m, loader, device, domain_name, mode='seed'):
    all_pred, all_true = [], []
    with torch.no_grad():
        for X, y in loader:
            X = X.to(device)
            if mode == 'seed':
                logits, _ = m(x_seed=X, mode='seed')
            else:
                logits, _ = m(x_brain=X, mode='brain')
            all_pred.extend(logits.argmax(1).cpu().numpy())
            all_true.extend(y.numpy())
    
    print(f'\n=== {domain_name} Test Results ===')
    print(classification_report(all_true, all_pred,
                                target_names=['Class 0', 'Class 1', 'Class 2'],
                                zero_division=0))
    return accuracy_score(all_true, all_pred)

model.eval()
seed_final = full_eval(m_ref, loader_seed_test, DEVICE, 'SEED (Source Domain)', mode='seed')
brain_final = full_eval(m_ref, loader_brain_test, DEVICE, 'Brainwave (Target Domain)', mode='brain')

print('\n' + '='*80)
print(f'{"DG-BDH Cross-Domain Transfer Results":^80}')
print('='*80)
print(f' Training: SEED Dataset (62 channels × {cfg.SEED_FEAT_DIM} features)')
print(f' Testing:  Brainwave Dataset (8 channels × {cfg.BRAIN_FEAT_DIM} features)')
print('-'*80)
print(f' SEED Test Accuracy (source):       {seed_final:.4f}')
print(f' Brainwave Test Accuracy (target):  {brain_final:.4f}')
print(f' Best SEED Accuracy:                {best_seed_acc:.4f}')
print(f' Best Brainwave Accuracy:           {best_brain_acc:.4f}')
print('='*80)
print(f'\nBAH Module Effectiveness Metrics:')
print(f' Domain Centroid Distance Before BAH: {domain_distance_before:.4f}')
print(f' Domain Centroid Distance After BAH:  {domain_distance_after:.4f}')
print(f' Reduction: {reduction_pct:.2f}%')
print(f'\n MMD Before BAH: {mmd_before:.4f}')
print(f' MMD After BAH:  {mmd_after:.4f}')
print(f' Reduction: {mmd_reduction_pct:.2f}%')
print('='*80)
print('\nQualitative visualizations generated:')
print(' 1. dgbdh_comprehensive_qualitative_analysis.png')
print(' 2. dgbdh_tsne_bah_effectiveness.png')
print(' 3. dgbdh_tsne_classwise_alignment.png')
print(' 4. dgbdh_seed_to_brainwave.png')
print('\nCross-domain transfer complete!')